In [1]:
# Monkey-patch to get live output from python file into this notebook
import subprocess
import sys

_original_run = subprocess.run

def notebook_run(args, **kwargs):
    process = subprocess.Popen(
        [str(arg) for arg in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

    return subprocess.CompletedProcess(args, return_code)

subprocess.run = notebook_run


In [2]:
%load_ext autoreload
%autoreload 2

import importlib
import src.combine
importlib.reload(src.combine)

from src.combine import run_colmap
import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
# run colmap
runit = run_colmap(image_dir="inputs/water_tower_converted", output_dir="outputs/water_tower")
#runit.run()   # this takes a long time to run, just use old outputs for testing changes to the run_colmap object

I20260508 16:06:42.882245 46052 feature_extraction.cc:504] === Feature extraction ===
I20260508 16:06:42.883997 34048 sift.cc:757] Creating SIFT GPU feature extractor
I20260508 16:06:43.372999 38188 feature_extraction.cc:282] Processed file [1/40]
I20260508 16:06:43.373150 38188 feature_extraction.cc:285]   Name:            DJI_20260509041936_0191_D.jpg
I20260508 16:06:43.373195 38188 feature_extraction.cc:294]   Dimensions:      2000 x 1500
I20260508 16:06:43.373237 38188 feature_extraction.cc:297]   Camera:          #1 - SIMPLE_RADIAL
I20260508 16:06:43.373272 38188 feature_extraction.cc:300]   Focal Length:    2400.00px
I20260508 16:06:43.373325 38188 feature_extraction.cc:304]   Features:        21469 (SIFT)
I20260508 16:06:43.529825 38188 feature_extraction.cc:282] Processed file [2/40]
I20260508 16:06:43.529889 38188 feature_extraction.cc:285]   Name:            DJI_20260509041941_0192_D.jpg
I20260508 16:06:43.529898 38188 feature_extraction.cc:294]   Dimensions:      2000 x 1500

In [5]:
# visualize
ply_path = "outputs/coy/dense/fused.ply"

pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])

PointCloud with 1336848 points.
Number of points: 1336848


In [9]:
# remove noise
clean_pcd, noise_idx = runit.remove_noise(30, 1)
clean_pcd.paint_uniform_color([1, 0.706, 0])

# select noise
noise = pcd.select_by_index(noise_idx, invert=True)
noise.paint_uniform_color([1, 0, 0])

# visualize
o3d.visualization.draw_geometries([clean_pcd, noise])

In [11]:
subsampled_pcd = runit.adaptive_subsample()
o3d.visualization.draw_geometries([clean_pcd])
print("Number of points:", len(subsampled_pcd.points))
print(f"Subsampling reduced number of points by {100 * (1 - len(subsampled_pcd.points) / len(pcd.points))}%")

Number of points: 348377
Subsampling reduced number of points by 74.02693355182767%


In [13]:
features = runit.extract_features()
features

{'fpfh': array([[ 2.97372844,  0.11223156,  0.09349094, ...,  3.7930437 ,
          2.97607519,  0.70705312],
        [ 0.17569209,  0.1827549 ,  0.15670286, ...,  7.27118638,
          6.29274027, 16.7387379 ],
        [ 5.23231579,  0.10309396,  0.08498895, ...,  4.27317116,
          3.0057065 ,  0.6507244 ],
        ...,
        [ 0.        ,  0.        ,  0.        , ..., 20.2215507 ,
         40.00070957,  0.15859358],
        [ 0.        ,  0.        ,  0.        , ..., 22.05852676,
          0.        ,  6.07794665],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]], shape=(37861, 33)),
 'distance_to_centroid': array([ 7.43635295,  7.37296534,  7.43901746, ...,  7.2208682 ,
         6.46729558, 10.40098148], shape=(37861,)),
 'height': array([ 8.46153641,  8.27970886,  8.45790195, ..., 12.90998268,
        12.29872322, 14.82501316], shape=(37861,))}

In [5]:
mesh = runit.point_cloud_to_mesh()
mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([mesh])

In [9]:
voxels = runit.voxelize(0.1)
o3d.visualization.draw_geometries([voxels])